In [ ]:
def generate_headers():
    adb_secret_scope = os.environ.get("KEY_VAULT_SCOPE_NAME");
    api_token = dbutils.secrets.get(scope = adb_secret_scope, key = "mxdata-service-aad-token")
    header = {"Authorization": f"Bearer {api_token}"}
    return header

def get(url):
    response = requests.get(
        url,
        headers=generate_headers()
        )
    return response

def post(url, payload):
    response = requests.post(
        url,
        data=json.dumps(payload),
        headers=generate_headers()
        )
    return response

def patch(url, payload):
    response = requests.patch(
        url,
        data=json.dumps(payload),
        headers=generate_headers()
        )
    return response

def check_existing_ncc(ncc_name, ncc_url):
    current_nccs = get(ncc_url).json()
    for current_ncc in current_nccs:
        if current_ncc.get("name") == ncc_name:
            return current_ncc.get("network_connectivity_config_id")
        else:
            return None

def create_ncc(ncc_name, ncc_region, account_url):
    ncc_url = f"{account_url}/network-connectivity-configs"
    ncc_id = check_existing_ncc(ncc_name, ncc_url)
    if ncc_id is None:
        ncc_payload = {
            "name": ncc_name,
            "region": ncc_region
        }
        ncc_id = post(ncc_url, ncc_payload).json().get("network_connectivity_config_id")
    return ncc_id

def add_endpoints(sa_resource_ids, group_id, ncc_id, account_url):
    url = f"{account_url}/network-connectivity-configs/{ncc_id}/private-endpoint-rules"
    for sa_resource_id in sa_resource_ids:
        payload = {
            "resource_id": sa_resource_id,
            "group_id": group_id
        }
        post(url, payload)

def provision_workspaces(workspace_ids, ncc_id, account_url):
    for workspace_id in workspace_ids:
        workspace_url = f"{account_url}/workspaces/{workspace_id}"
        payload = {
            "network_connectivity_config_id": ncc_id
        }
        print(patch(workspace_url, payload))

def main(ncc_name, ncc_region, sa_resource_ids, workspace_ids):
    notebook_context = json.loads(dbutils.notebook.entry_point.getDbutils().notebook().getContext().toJson())
    account_id = notebook_context.get("tags").get("accountId")
    account_url = f"https://accounts.azuredatabricks.net/api/2.0/accounts/{account_id}"

    ncc_id = create_ncc(ncc_name, ncc_region, account_url)
    add_endpoints(sa_resource_ids, "blob", ncc_id, account_url)
    add_endpoints(sa_resource_ids, "dfs", ncc_id, account_url)
    provision_workspaces(workspace_ids, ncc_id, account_url)


In [ ]:
import requests
import json
import os

ncc_name = ""
ncc_region = ""

# Checking if private endpoint rules exist based on storage account resource id is not possible
# To avoid duplicate private endpoint creation delete created resource ids
sa_resource_ids = [

]

workspace_ids = [

]

main(ncc_name, ncc_region, sa_resource_ids, workspace_ids)
